# Build the TREC 2023 Corpus + Topics (external held-out test)

Foundation for evaluating the **frozen** pipeline on TREC 2023 Clinical Trials — a genuinely unseen
external test (no component of ours ever saw it). Produces a full-text corpus aligned the same way as
the 2022 corpus, plus patient-text derived from the questionnaire topics via a **fixed, blind template**.

**Caveats (honest):** (1) different corpus snapshot (2023-05-08, ~450k trials) — a full rebuild;
(2) **questionnaire topic format** (structured fields, not vignettes) — a domain shift our pipeline is
not tuned for, so this is a *generalization stress test*. The questionnaire→text template is defined
here **without looking at qrels** (no test peeking).

**Outputs (Drive):** `doc_texts_fulltext_2023.txt`, `index2docid_2023.txt`, `topics2023_text.jsonl`.
Downstream: `eval_external_2023.ipynb` (retrieve → 9 features → frozen LambdaMART → trec_eval).

In [ ]:
!pip install -q lxml tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
DATA_ROOT = '/content/drive/MyDrive/ct_data23'
OUT_DIR   = f'{DATA_ROOT}/trec2023'
os.makedirs(OUT_DIR, exist_ok=True)
BASE = 'https://trec-cds.org'
CORPUS_ZIPS = [f'{BASE}/2023_data/ClinicalTrials.2023-05-08.trials{i}.zip' for i in range(6)]
TOPICS_URL  = f'{BASE}/topics2023.xml'
QRELS_URL   = 'https://trec.nist.gov/data/trials/qrels2023.txt'   # verified public (37 judged topics)
print('config set')

In [ ]:
# Download corpus zips + topics into a scratch dir (not Drive — 2.1 GB).
SCRATCH = '/content/ct2023'
os.makedirs(SCRATCH, exist_ok=True)
for url in CORPUS_ZIPS:
    fn = f"{SCRATCH}/{url.split('/')[-1]}"
    if not os.path.exists(fn):
        !wget -q -O "$fn" "$url" && echo "got {url.split('/')[-1]}"
!wget -q -O "$SCRATCH/topics2023.xml" "$TOPICS_URL" && echo 'got topics'
print('downloaded:', os.listdir(SCRATCH))

In [ ]:
import zipfile, glob
from lxml import etree
from tqdm.auto import tqdm

def _txt(el):
    return ' '.join(el.itertext()).strip() if el is not None else ''

def parse_trial(xml_bytes):
    """Classic ClinicalTrials.gov XML -> (nct_id, full_text). Mirrors the 2022 field assembly."""
    try:
        r = etree.fromstring(xml_bytes)
    except Exception:
        return None, None
    nct = r.findtext('.//id_info/nct_id')
    if not nct:
        return None, None
    parts = []
    for path in ['brief_title', 'official_title']:
        t = r.findtext(path)
        if t and t.strip(): parts.append(t.strip())
    conds = [c.text.strip() for c in r.findall('condition') if c.text]
    if conds: parts.append('Conditions: ' + '; '.join(conds))
    for path in ['brief_summary', 'detailed_description']:
        t = _txt(r.find(path))
        if t: parts.append(t)
    ivs = []
    for iv in r.findall('intervention'):
        nm = iv.findtext('intervention_name'); ty = iv.findtext('intervention_type')
        if nm: ivs.append(f"{(ty or '').strip()} {nm.strip()}".strip())
    if ivs: parts.append('Interventions: ' + '; '.join(ivs))
    elig = _txt(r.find('eligibility/criteria'))
    if elig: parts.append('Eligibility: ' + elig)
    return nct.strip(), '\n'.join(parts)

ids, texts = [], []
for zf in sorted(glob.glob(f'{SCRATCH}/ClinicalTrials.2023-05-08.trials*.zip')):
    with zipfile.ZipFile(zf) as z:
        names = [n for n in z.namelist() if n.endswith('.xml')]
        for n in tqdm(names, desc=os.path.basename(zf)):
            nct, txt = parse_trial(z.read(n))
            if nct:
                ids.append(nct); texts.append(txt.replace('\n', ' ').replace('\r', ' '))
print(f'parsed {len(ids):,} trials')

In [ ]:
with open(f'{OUT_DIR}/index2docid_2023.txt', 'w') as f:
    f.write('\n'.join(ids) + '\n')
with open(f'{OUT_DIR}/doc_texts_fulltext_2023.txt', 'w') as f:
    f.write('\n'.join(texts) + '\n')
import numpy as np
print(f'saved corpus: {len(ids):,} docs -> {OUT_DIR}')
print('median doc chars:', int(np.median([len(t) for t in texts[:5000]])))

## Questionnaire → patient-text (blind, fixed template)
Each `<topic template=disorder>` with `<field name>value</field>` entries becomes a prose-like
description. Fixed rule, defined without qrels: `"A patient with {disorder}. {Field}: {value}. ..."`,
skipping empty fields. No test peeking.

In [ ]:
import json
root = etree.parse(f'{SCRATCH}/topics2023.xml').getroot()
topics = []
for tp in root:
    tid = tp.get('number'); disorder = (tp.get('template') or '').replace('_', ' ')
    sent = [f'A patient with {disorder}.'] if disorder else []
    for fld in tp.findall('field'):
        name = (fld.get('name') or '').strip(); val = (fld.text or '').strip()
        if val:
            sent.append(f'{name.capitalize()}: {val}.')
    topics.append({'topic_id': tid, 'template': disorder, 'topic_text': ' '.join(sent)})
with open(f'{OUT_DIR}/topics2023_text.jsonl', 'w') as f:
    for t in topics:
        f.write(json.dumps(t) + '\n')
print(f'{len(topics)} topics -> {OUT_DIR}/topics2023_text.jsonl\n')
print('example:', topics[0]['topic_text'])
print('example:', topics[5]['topic_text'])

In [ ]:
# Fetch the 2023 qrels (public) to Drive so the eval notebook has them.
import urllib.request
qpath = f'{OUT_DIR}/qrels2023.txt'
urllib.request.urlretrieve(QRELS_URL, qpath)
njudged = sum(1 for _ in open(qpath))
ntopics = len({l.split()[0] for l in open(qpath)})
print(f'qrels -> {qpath}  ({njudged:,} pairs, {ntopics} judged topics)')

## Next: `eval_external_2023.ipynb` (to build)

With the corpus + topic-text in hand, the external eval mirrors the frozen 2022 pipeline:
1. BM25 over `doc_texts_fulltext_2023`; dense = re-encode this corpus with `ctmatch-retriever-v2` (~1 hr GPU).
2. Hybrid RRF pool; extract the 9 features (clf-v4, reranker-v2, Qwen `llm_yesno` top-500).
3. Score with the **frozen** `ensemble_full_v1.txt` LambdaMART booster — no retraining, no tuning.
4. `pytrec_eval` vs the 2023 qrels (set `QRELS_URL` once located). Report once; this is a held-out test.

Everything is frozen — the only 2023-specific inputs are the corpus and the blind questionnaire template.